## 1.

In [ ]:
!pip install openai runwayml opencv-python-headless torch torchvision
!pip install git+https://github.com/openai/CLIP.git
!pip install pandas matplotlib requests Pillow

## 2. Config & API Keys

In [ ]:
import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')   # промпты
RUNWAY_API_KEY = userdata.get('RUNWAY_API_KEY')   # API Keys

os.makedirs('outputs/videos', exist_ok=True)
os.makedirs('outputs/frames', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

print('✅ Config ready')

## 3. Prompt Generator (LLM)

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_STRATEGIES = {
    'descriptive': (
        'Convert this topic into a vivid, cinematic video prompt (max 200 chars).\n'
        'Focus on visual elements: lighting, movement, environment.\n'
        'Topic: {topic}'
    ),
    'storytelling': (
        'Create a video prompt that tells a visual story about this topic.\n'
        'Include: scene, subject, action, mood. Max 200 chars.\n'
        'Topic: {topic}'
    ),
    'educational': (
        'Create a clear, educational video prompt visualizing this concept.\n'
        'Make it engaging and easy to understand visually. Max 200 chars.\n'
        'Topic: {topic}'
    ),
}

def generate_video_prompt(topic: str, strategy: str = 'descriptive') -> str:
    template = PROMPT_STRATEGIES[strategy]
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': template.format(topic=topic)}],
        max_tokens=200
    )
    return response.choices[0].message.content.strip()

def improve_prompt(original_prompt: str, feedback: str) -> str:
    """Auto-improve prompt based on quality feedback."""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content':
            f'Improve this video generation prompt based on feedback.\n'
            f'Original: {original_prompt}\n'
            f'Feedback: {feedback}\n'
            f'Return only the improved prompt, max 200 chars.'
        }],
        max_tokens=200
    )
    return response.choices[0].message.content.strip()

# Test
test_prompt = generate_video_prompt('photosynthesis in plants', 'educational')
print(f'Generated prompt:\n{test_prompt}')

## 4. Video Generator (Runway ML)

In [ ]:
import requests
import time

def generate_video(prompt: str, output_path: str, duration: int = 5) -> str | None:
    """
    Generate video via Runway Gen-3 API.
    Returns path to downloaded video or None on failure.
    """
    headers = {
        'Authorization': f'Bearer {RUNWAY_API_KEY}',
        'Content-Type': 'application/json',
        'X-Runway-Version': '2024-11-06'
    }

    # Таск
    response = requests.post(
        'https://api.dev.runwayml.com/v1/image_to_video',
        headers=headers,
        json={
            'model': 'gen3a_turbo',
            'promptText': prompt,
            'duration': duration,
            'ratio': '1280:768'
        }
    )

    if response.status_code != 200:
        print(f'❌ Generation failed: {response.text}')
        return None

    task_id = response.json()['id']
    print(f'⏳ Task submitted: {task_id}')

    # Poll for completion
    for attempt in range(30):
        time.sleep(10)
        status_response = requests.get(
            f'https://api.dev.runwayml.com/v1/tasks/{task_id}',
            headers=headers
        )
        status = status_response.json()

        if status['status'] == 'SUCCEEDED':
            video_url = status['output'][0]
            video_data = requests.get(video_url).content
            with open(output_path, 'wb') as f:
                f.write(video_data)
            print(f'✅ Video saved: {output_path}')
            return output_path

        elif status['status'] == 'FAILED':
            print(f'❌ Task failed: {status.get("error")}')
            return None

        print(f'  [{attempt+1}/30] Status: {status["status"]}...')

    print('❌ Timeout')
    return None

## 5. Quality Evaluator

In [ ]:
import cv2
import clip
import torch
import numpy as np
from PIL import Image
from dataclasses import dataclass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, clip_preprocess = clip.load('ViT-B/32', device=device)

@dataclass
class VideoQuality:
    clip_score: float       # соответствие промпту (0-1)
    motion_score: float     # движение/действия (0-1)
    sharpness: float        # качество (0-1)
    overall: float          # weighted average
    feedback: str           # human-readable feedback

def extract_frames(video_path: str, n_frames: int = 8) -> list:
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def compute_clip_score(prompt: str, frames: list) -> float:
    """Measure alignment between prompt text and video frames."""
    text = clip.tokenize([prompt]).to(device)
    scores = []
    for frame in frames:
        img = clip_preprocess(Image.fromarray(frame)).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = clip_model.encode_image(img)
            text_features = clip_model.encode_text(text)
            similarity = torch.cosine_similarity(image_features, text_features).item()
        scores.append(similarity)
    return float(np.mean(scores))

def compute_motion_score(frames: list) -> float:
    """Optical flow-based motion estimation."""
    if len(frames) < 2:
        return 0.0
    scores = []
    for i in range(len(frames) - 1):
        prev = cv2.cvtColor(frames[i], cv2.COLOR_RGB2GRAY)
        curr = cv2.cvtColor(frames[i+1], cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prev, curr, None,
                    0.5, 3, 15, 3, 5, 1.2, 0)
        magnitude = np.sqrt(flow[...,0]**2 + flow[...,1]**2)
        scores.append(np.mean(magnitude))
    # Normalize: 0-20 pixel movement → 0-1
    return float(min(np.mean(scores) / 20.0, 1.0))

def compute_sharpness(frames: list) -> float:
    """Laplacian variance — higher = sharper frames."""
    scores = []
    for frame in frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        variance = cv2.Laplacian(gray, cv2.CV_64F).var()
        scores.append(variance)
    # Normalize: cap at 1000
    return float(min(np.mean(scores) / 1000.0, 1.0))

def evaluate_video(video_path: str, prompt: str) -> VideoQuality:
    print(f'🔍 Evaluating: {video_path}')
    frames = extract_frames(video_path)

    clip_score = compute_clip_score(prompt, frames)
    motion_score = compute_motion_score(frames)
    sharpness = compute_sharpness(frames)

    # Weighted overall score
    overall = 0.5 * clip_score + 0.3 * motion_score + 0.2 * sharpness

    # Generate feedback for prompt improvement
    issues = []
    if clip_score < 0.25:  issues.append('low prompt alignment — be more specific')
    if motion_score < 0.2:  issues.append('video is too static — add movement/action verbs')
    if sharpness < 0.3:     issues.append('frames are blurry — describe clearer visual details')
    feedback = '; '.join(issues) if issues else 'quality looks good'

    result = VideoQuality(
        clip_score=round(clip_score, 3),
        motion_score=round(motion_score, 3),
        sharpness=round(sharpness, 3),
        overall=round(overall, 3),
        feedback=feedback
    )
    print(f'  CLIP: {result.clip_score} | Motion: {result.motion_score} | '
          f'Sharpness: {result.sharpness} | Overall: {result.overall}')
    print(f'  Feedback: {result.feedback}')
    return result

print('✅ Evaluator ready')

## 6. Full Pipeline with Auto-Improvement Loop

In [ ]:
import pandas as pd

def run_pipeline(
    topic: str,
    strategy: str = 'educational',
    max_iterations: int = 3,
    quality_threshold: float = 0.4
) -> dict:
    """
    Full pipeline:
    1. Generate prompt from topic
    2. Generate video
    3. Evaluate quality
    4. If quality < threshold → improve prompt and retry
    """
    print(f'\n{'='*50}')
    print(f'🚀 Topic: {topic} | Strategy: {strategy}')
    print(f'{'='*50}')

    results = []
    prompt = generate_video_prompt(topic, strategy)

    for iteration in range(1, max_iterations + 1):
        print(f'\n--- Iteration {iteration}/{max_iterations} ---')
        print(f'Prompt: {prompt}')

        video_path = f'outputs/videos/{topic[:20].replace(" ","_")}_iter{iteration}.mp4'
        video_path = generate_video(prompt, video_path)

        if not video_path:
            print('⚠️ Skipping evaluation — video generation failed')
            break

        quality = evaluate_video(video_path, prompt)
        results.append({
            'topic': topic,
            'strategy': strategy,
            'iteration': iteration,
            'prompt': prompt,
            'video_path': video_path,
            'clip_score': quality.clip_score,
            'motion_score': quality.motion_score,
            'sharpness': quality.sharpness,
            'overall': quality.overall,
            'feedback': quality.feedback,
        })

        if quality.overall >= quality_threshold:
            print(f'✅ Quality threshold reached ({quality.overall:.3f} >= {quality_threshold})')
            break
        elif iteration < max_iterations:
            print(f'⚠️ Quality low ({quality.overall:.3f}). Improving prompt...')
            prompt = improve_prompt(prompt, quality.feedback)

    return results

print('✅ Pipeline ready')

## 7. Run Experiment

In [ ]:
# Темы и стратегии
TOPICS = [
    'photosynthesis in a forest',
    'neurons firing in the brain',
    'water cycle in nature',
]

STRATEGIES = ['descriptive', 'storytelling', 'educational']

all_results = []

topic = TOPICS[0]
for strategy in STRATEGIES:
    results = run_pipeline(topic, strategy=strategy, max_iterations=2)
    all_results.extend(results)

df = pd.DataFrame(all_results)
df.to_csv('outputs/results/experiment_results.csv', index=False)
print('\n📊 Results saved!')
df

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv('outputs/results/experiment_results.csv')

final = df.groupby('strategy')[['clip_score', 'motion_score', 'sharpness', 'overall']].mean().round(3)
print('Strategy comparison:')
print(final.to_string())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Prompt Strategy Comparison — Video Quality', fontsize=13, fontweight='bold')

metrics = ['clip_score', 'motion_score', 'sharpness', 'overall']
colors = ['#4C9BE8', '#E87C4C', '#4CE8A0', '#A04CE8']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(final.index, final[metric], color=color, alpha=0.85)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_ylim(0, 1)
    ax.set_xticklabels(final.index, rotation=20, ha='right', fontsize=9)
    ax.bar_label(bars, fmt='%.3f', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/results/strategy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved!')

## 9. Show Improvement Loop Results

In [ ]:
# Итерации и качествл
if df['iteration'].max() > 1:
    fig, ax = plt.subplots(figsize=(8, 4))
    for strategy in df['strategy'].unique():
        subset = df[df['strategy'] == strategy].groupby('iteration')['overall'].mean()
        ax.plot(subset.index, subset.values, marker='o', label=strategy)
    ax.set_title('Quality Improvement Across Iterations')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Overall Quality Score')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('outputs/results/improvement_loop.png', dpi=150)
    plt.show()
else:
    print('Run with max_iterations=3 to see improvement loop chart')